### Import libraries

In [1]:

from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


### Set paths

In [ ]:

ROOT = Path(
    r"D:\My_Project\Analyse\AI-Paddy-Farming-Intelligence"
)

PROCESSED_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models"
EVALUATION_DIR = ROOT / "evaluation"

EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "rice_disease_model.pth"
TEST_PATH = PROCESSED_DIR / "test.csv"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


### Load model and class names

In [ ]:

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=True
)

class_to_idx = checkpoint["class_to_idx"]

idx_to_class = {
    index: name
    for name, index in class_to_idx.items()
}

num_classes = len(class_to_idx)

model = efficientnet_b0(weights=None)

in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features,
    num_classes
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

print("Model loaded successfully!")
print("Classes:", idx_to_class)


### Load test images

In [ ]:

test_df = pd.read_csv(TEST_PATH)

IMAGE_SIZE = checkpoint["image_size"]

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=checkpoint["mean"],
        std=checkpoint["std"]
    )
])


class RiceTestDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = ROOT / row["image_path"]

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        image = test_transform(image)

        label = class_to_idx[row["label"]]

        return image, label


test_dataset = RiceTestDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Test images:", len(test_dataset))


### Predict all test images

In [ ]:

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

print("Prediction completed!")


### Calculate model performance

In [ ]:

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

recall = recall_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

print("MODEL PERFORMANCE")
print("-------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")


### Classification report

In [ ]:

class_names = [
    idx_to_class[i]
    for i in range(num_classes)
]

report = classification_report(
    all_labels,
    all_predictions,
    labels=list(range(num_classes)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

display(report_df)

report_df.to_csv(
    EVALUATION_DIR / "classification_report.csv"
)


### Confusion matrix

In [ ]:

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=list(range(num_classes))
)

fig, ax = plt.subplots(figsize=(11, 9))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

display.plot(
    cmap="Blues",
    xticks_rotation=45,
    ax=ax,
    colorbar=False
)

plt.title("Rice Leaf Disease Classification")
plt.tight_layout()
plt.show()

pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
).to_csv(EVALUATION_DIR / "confusion_matrix.csv")

print("Evaluation results saved!")
